<a href="https://colab.research.google.com/github/bivar/llm_studies/blob/main/autonomous_eda_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autonomous EDA Agent

## Config

In [1]:
!pip install -U langchain langchain-experimental langchain-google-genai pandas matplotlib seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.4/236.4 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successful

## Model Selection

In [9]:
import pandas as pd
import numpy as np
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

api_key = userdata.get('GEMINI_API_KEY')

# using most recent gemini flash model (flash is the best option when using tool calling)
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=api_key,
    temperature=0
)

## Fake Data Gathering

In [10]:
import pandas as pd
import numpy as np

np.random.seed(42)

jogadoras = ['Gabi Guimarães', 'Thaisa Daher', 'Rosamaria Montibeller', 'Macris Carneiro',
             'Nyeme Costa', 'Ana Cristina', 'Carolana', 'Roberta Ratzke',
             'Kisy Nascimento', 'Julia Bergmann', 'Diana Duarte', 'Pri Daroit']

posicoes = ['Ponteira', 'Central', 'Oposta', 'Levantadora',
            'Líbero', 'Ponteira', 'Central', 'Levantadora',
            'Oposta', 'Ponteira', 'Central', 'Ponteira']

# fake data
dados_volei = {
    'Jogadora': jogadoras,
    'Posicao': posicoes,
    'Idade': [29, 36, 29, 35, 25, 20, 32, 33, 24, 23, 25, 35],
    'Partidas_Jogadas': np.random.randint(15, 35, size=12),
    'Pontos_Ataque': np.random.randint(80, 400, size=12),
    'Pontos_Bloqueio': np.random.randint(10, 90, size=12),
    'Aces': np.random.randint(0, 35, size=12),
    'Eficiencia_Ataque_Pct': np.round(np.random.uniform(35.0, 60.0, size=12), 2),
    'Titular': [True, True, True, True, True, False, True, False, False, False, False, False]
}

df_volei = pd.DataFrame(dados_volei)

# business rule: libero doesnt attack, serve or block
is_libero = df_volei['Posicao'] == 'Líbero'
df_volei.loc[is_libero, ['Pontos_Ataque', 'Pontos_Bloqueio', 'Aces', 'Eficiencia_Ataque_Pct']] = 0

# featuring engineering
df_volei['Pontos_Totais'] = df_volei['Pontos_Ataque'] + df_volei['Pontos_Bloqueio'] + df_volei['Aces']

display(df_volei.head())

,Jogadora,Posicao,Idade,Partidas_Jogadas,Pontos_Ataque,Pontos_Bloqueio,Aces,Eficiencia_Ataque_Pct,Titular,Pontos_Totais
0,Gabi Guimarães,Ponteira,29,21,229,68,2,41.03,True,299
1,Thaisa Daher,Central,36,34,388,51,6,52.08,True,445
2,Rosamaria Montibeller,Oposta,29,29,337,69,20,50.25,True,426
3,Macris Carneiro,Levantadora,35,25,373,89,8,55.83,True,470
4,Nyeme Costa,Líbero,25,22,0,0,0,0.00,True,0


## Agent Architecture

In [11]:
agent = create_pandas_dataframe_agent(
    llm=llm,
    df=df_volei,
    verbose=True, # thinking activated
    allow_dangerous_code=True,
    agent_type="tool-calling" # tool calling is the most stable version
)

## Testing

In [13]:
resposta = agent.invoke("Qual a jogadora com a maior eficiência de ataque?")
print(resposta['output'])



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': '# Encontrar a linha com a maior eficiência de ataque\nmax_eficiencia_idx = df[\'Eficiencia_Ataque_Pct\'].idxmax()\njogadora_maior_eficiencia = df.loc[max_eficiencia_idx, \'Jogadora\']\nvalor_eficiencia = df.loc[max_eficiencia_idx, \'Eficiencia_Ataque_Pct\']\n\nprint(f"Jogadora: {jogadora_maior_eficiencia}")\nprint(f"Eficiência: {valor_eficiencia}")\n'}`


Jogadora: Macris Carneiro
Eficiência: 55.83

Invoking: `python_repl_ast` with `{'query': "# Corrigindo a busca para exibir o nome da jogadora corretamente\njogadora_maior_eficiencia = df.loc[df['Eficiencia_Ataque_Pct'].idxmax(), 'Jogadora']\nprint(jogadora_maior_eficiencia)\n"}`


Macris Carneiro
[{'type': 'text', 'text': 'A jogadora com a maior eficiência de ataque é **Macris Carneiro**, com uma eficiência de **55.83%**.', 'index': 0, 'extras': {'signature': 'EjQKMgEMOdbHHl1sQLzQn6RXyWXTx5FYCB8HjCu1NeWOD2KekQzpXUMes2jtjbALXwXlFXd0'}}]

> Finished ch